# 🧬 PHGDH × Alzheimer's Disease — Comprehensive Target Analysis

**Gene:** PHGDH (Phosphoglycerate Dehydrogenase) — *ENSG00000092621*  
**Disease:** Alzheimer's Disease — *MONDO_0004975*  
**Context:** Emerging target highlighted in *Cell* 2025  

### What this notebook covers
| Section | Data Source | What you get |
|---------|------------|-------------|
| 1 | Open Targets | Genetic Association, Expression, Tractability (GET) scores |
| 2 | Open Targets | All datatype evidence scores (full breakdown) |
| 3 | Open Targets | Known drugs & tractability labels |
| 4 | PubTator3 | Gene mention co-occurrence + publication velocity |
| 5 | Europe PMC | Literature counts, top papers, velocity trend |
| 6 | ClinicalTrials.gov | Trial count, phases, active trials |
| 7 | Summary Dashboard | Combined GET score + all stats in one view |


---
## ⚙️ Setup — Install dependencies

In [ ]:
# Run once — installs required libraries
!pip install requests pandas matplotlib seaborn ipywidgets --quiet
print('✅ Dependencies ready')

In [ ]:
import requests
import json
import math
import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── Constants ──────────────────────────────────────────────────────────────
GENE_SYMBOL   = 'PHGDH'
GENE_ENSEMBL  = 'ENSG00000092621'
DISEASE_NAME  = 'Alzheimer disease'
DISEASE_EFO   = 'MONDO_0004975'          # Alzheimer disease
DISEASE_EFO2  = 'EFO_0000249'            # fallback

OT_API        = 'https://api.platform.opentargets.org/api/v4/graphql'
CT_API        = 'https://clinicaltrials.gov/api/v2/studies'
EPMC_API      = 'https://www.ebi.ac.uk/europepmc/webservices/rest/search'
PUBMED_API    = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
PUBTATOR_API  = 'https://www.ncbi.nlm.nih.gov/research/pubtator3-api'

HEADERS       = {'Content-Type': 'application/json'}

print(f'🎯 Target gene : {GENE_SYMBOL} ({GENE_ENSEMBL})')
print(f'🧠 Disease     : {DISEASE_NAME} ({DISEASE_EFO})')

---
## 🎯 Section 1 — Open Targets: GET Scores

Reproduces the exact scoring logic from `api.ts`:
- **G (Genetic)** = max(genetic_association, somatic_mutation, genetic_literature)
- **E (Expression)** = strength × 0.70 + selectivity × 0.30  (log10-normalised RNA)
- **T (Tractability)** = tractability hierarchy score
- **GET** = G × 0.50 + E × 0.25 + T × 0.25

In [ ]:
def fetch_ot_target_disease(ensembl_id: str, efo_id: str):
    """Fetch PHGDH association data from Open Targets for a given disease EFO."""
    query = """
    query GetTargetDisease($ensemblId: String!, $efoId: String!) {
      target(ensemblId: $ensemblId) {
        id
        approvedSymbol
        approvedName
        biotype
        tractability {
          label
          modality
          value
        }
        expressions {
          tissue { label }
          rna { value }
          protein { level }
        }
      }
      disease(efoId: $efoId) {
        id
        name
        associatedTargets(page: {index: 0, size: 300}) {
          rows {
            target { id approvedSymbol }
            score
            datatypeScores { id score }
          }
        }
      }
    }
    """
    resp = requests.post(
        OT_API,
        headers=HEADERS,
        json={'query': query, 'variables': {'ensemblId': ensembl_id, 'efoId': efo_id}},
        timeout=30
    )
    return resp.json()


def compute_expression_score(expressions: list) -> float:
    """Mirrors api.ts expressionScore logic."""
    vals = sorted(
        [e['rna']['value'] for e in expressions if e.get('rna') and e['rna'].get('value', 0) > 0],
        reverse=True
    )
    if not vals:
        return 0.0
    top1    = vals[0]
    top3avg = sum(vals[:3]) / min(3, len(vals))
    mean_all = sum(vals) / len(vals)
    strength    = min(1.0, math.log10(top3avg + 1) / 4)
    selectivity = min(1.0, math.log10((top1 / mean_all) + 1) / math.log10(6)) if mean_all > 0 else 0
    return round(strength * 0.7 + selectivity * 0.3, 4)


def compute_tractability_score(tractability: list) -> float:
    """Mirrors api.ts targetScore hierarchy."""
    def has(modality, label):
        return any(t['modality'] == modality and t['label'] == label and t['value'] for t in tractability)
    if has('SM','Approved Drug') or has('AB','Approved Drug') or has('PR','Approved Drug'):
        return 1.00
    if has('SM','Advanced Clinical') or has('AB','Advanced Clinical') or has('PR','Advanced Clinical'):
        return 0.85
    if has('SM','Phase 1 Clinical') or has('AB','Phase 1 Clinical') or has('PR','Phase 1 Clinical'):
        return 0.70
    if has('SM','Structure with Ligand') or has('SM','High-Quality Ligand'):
        return 0.55
    if has('SM','High-Quality Pocket') or has('SM','Med-Quality Pocket'):
        return 0.40
    if has('SM','Druggable Family'):
        return 0.25
    return 0.10


# ── Fetch data ──────────────────────────────────────────────────────────────
print(f'🔍 Querying Open Targets for {GENE_SYMBOL} × {DISEASE_NAME}...')
data = fetch_ot_target_disease(GENE_ENSEMBL, DISEASE_EFO)

# Try fallback EFO if primary returns no association rows
rows = data.get('data', {}).get('disease', {}).get('associatedTargets', {}).get('rows', [])
if not rows:
    print(f'  ⚠️  No rows from {DISEASE_EFO}, trying fallback {DISEASE_EFO2}...')
    data = fetch_ot_target_disease(GENE_ENSEMBL, DISEASE_EFO2)
    rows = data.get('data', {}).get('disease', {}).get('associatedTargets', {}).get('rows', [])

target_data = data.get('data', {}).get('target', {})
disease_data = data.get('data', {}).get('disease', {})

print(f'  ✅ Target found : {target_data.get("approvedSymbol")} — {target_data.get("approvedName")}')
print(f'  ✅ Disease found: {disease_data.get("name")} ({disease_data.get("id")})')
print(f'  📊 Total disease-associated targets returned: {len(rows)}')

# ── Find PHGDH in the association rows ──────────────────────────────────────
phgdh_row = next((r for r in rows if r['target']['id'] == GENE_ENSEMBL), None)

if phgdh_row:
    datatype_scores = {s['id']: s['score'] for s in phgdh_row.get('datatypeScores', [])}
    overall_score   = phgdh_row.get('score', 0)
    print(f'\n  ✅ PHGDH found in Open Targets association list!')
    print(f'  Overall OT association score: {overall_score:.4f}')
else:
    print('\n  ⚠️  PHGDH not in top 300 association rows — computing scores directly from target data')
    datatype_scores = {}
    overall_score   = None

# ── Compute G / E / T ───────────────────────────────────────────────────────
G = max(
    datatype_scores.get('genetic_association', 0),
    datatype_scores.get('somatic_mutation', 0),
    datatype_scores.get('genetic_literature', 0)
)
E = compute_expression_score(target_data.get('expressions', []))
T = compute_tractability_score(target_data.get('tractability', []))
GET = round(G * 0.50 + E * 0.25 + T * 0.25, 4)

print(f'\n📊 GET Score Breakdown:')
print(f'   G (Genetic Association) = {G:.4f}')
print(f'   E (Expression)          = {E:.4f}')
print(f'   T (Tractability)        = {T:.4f}')
print(f'   ─────────────────────────────')
print(f'   GET Score               = {GET:.4f}  (G×0.50 + E×0.25 + T×0.25)')

---
## 📊 Section 2 — Open Targets: Full Datatype Evidence Breakdown

In [ ]:
# ── Full datatype scores table ───────────────────────────────────────────────
DATATYPE_LABELS = {
    'genetic_association' : 'Genetic Association',
    'somatic_mutation'    : 'Somatic Mutation',
    'known_drug'          : 'Known Drug',
    'affected_pathway'    : 'Affected Pathway',
    'literature'          : 'Literature Mining',
    'rna_expression'      : 'RNA Expression',
    'animal_model'        : 'Animal Model',
    'genetic_literature'  : 'Genetic Literature',
}

dt_rows = []
for k, label in DATATYPE_LABELS.items():
    score = datatype_scores.get(k, 0)
    dt_rows.append({'Evidence Type': label, 'ID': k, 'Score': round(score, 4)})

df_dt = pd.DataFrame(dt_rows).sort_values('Score', ascending=False)
print(f'\n📋 Open Targets Evidence Scores for {GENE_SYMBOL} in {DISEASE_NAME}\n')
display(df_dt.to_string(index=False))

# ── Bar chart ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2196F3' if s > 0 else '#E0E0E0' for s in df_dt['Score']]
bars = ax.barh(df_dt['Evidence Type'], df_dt['Score'], color=colors, edgecolor='white')
ax.set_xlabel('Score (0–1)', fontsize=11)
ax.set_title(f'Open Targets Evidence Scores: {GENE_SYMBOL} × {DISEASE_NAME}', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1)
for bar, val in zip(bars, df_dt['Score']):
    if val > 0:
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()
print(f'\n📌 Overall OT Association Score: {overall_score}')

---
## 💊 Section 3 — Open Targets: Tractability & Known Drugs

In [ ]:
# ── Tractability labels ──────────────────────────────────────────────────────
tractability = target_data.get('tractability', [])
active_tract = [t for t in tractability if t.get('value')]

print(f'\n🔬 Tractability Labels for {GENE_SYMBOL}:')
if active_tract:
    df_tract = pd.DataFrame(active_tract)[['modality', 'label', 'value']]
    df_tract.columns = ['Modality', 'Label', 'Active']
    display(df_tract)
else:
    print('  ⚠️  No active tractability labels found — target may be a novel/emerging target')

print(f'\n📊 Tractability Score = {T:.2f}')
tier_map = {
    1.00: 'Approved Drug',
    0.85: 'Advanced Clinical',
    0.70: 'Phase 1 Clinical',
    0.55: 'Structure with Ligand / High-Quality Ligand',
    0.40: 'High/Med-Quality Pocket',
    0.25: 'Druggable Family',
    0.10: 'No tractability evidence'
}
print(f'   Tier: {tier_map.get(T, "Unknown")}')

# ── Fetch known drugs ────────────────────────────────────────────────────────
drug_query = """
query GetDrugs($ensemblId: String!) {
  target(ensemblId: $ensemblId) {
    knownDrugs {
      rows {
        drug { id name maximumClinicalTrialPhase }
        status
        phase
        disease { name }
      }
    }
  }
}
"""
drug_resp = requests.post(OT_API, headers=HEADERS,
                          json={'query': drug_query, 'variables': {'ensemblId': GENE_ENSEMBL}},
                          timeout=20).json()
drug_rows = drug_resp.get('data', {}).get('target', {}).get('knownDrugs', {}).get('rows', [])

print(f'\n💊 Known Drugs targeting {GENE_SYMBOL}: {len(drug_rows)}')
if drug_rows:
    drugs_data = []
    seen = set()
    for r in drug_rows:
        did = r['drug']['id']
        if did not in seen:
            seen.add(did)
            drugs_data.append({
                'Drug Name' : r['drug']['name'],
                'Phase'     : r.get('phase'),
                'Status'    : r.get('status'),
                'Disease'   : r.get('disease', {}).get('name', 'N/A') if r.get('disease') else 'N/A',
            })
    df_drugs = pd.DataFrame(drugs_data)
    display(df_drugs)
else:
    print('  ℹ️  No known drugs on file — consistent with emerging target status')

---
## 🧫 Section 3b — Open Targets: Expression Profile (Brain Tissues)

In [ ]:
# ── Expression across tissues ────────────────────────────────────────────────
expressions = target_data.get('expressions', [])
exp_data = [
    {'Tissue': e['tissue']['label'], 'RNA Value': e['rna']['value']}
    for e in expressions
    if e.get('rna') and e['rna'].get('value', 0) > 0
]

df_exp = pd.DataFrame(exp_data).sort_values('RNA Value', ascending=False)
print(f'\n🧫 {GENE_SYMBOL} Expression: {len(df_exp)} tissues with RNA > 0')
print(f'   Expression Score (E) = {E:.4f}')

# Brain-relevant tissues
brain_keywords = ['brain', 'cerebr', 'hippocamp', 'cortex', 'neuron', 'cerebellum',
                  'frontal', 'temporal', 'parietal', 'occipital', 'amygdala', 'striatum']
df_brain = df_exp[df_exp['Tissue'].str.lower().str.contains('|'.join(brain_keywords), na=False)]

if not df_brain.empty:
    print(f'\n🧠 Brain-relevant tissues ({len(df_brain)} found):')
    display(df_brain.to_string(index=False))
else:
    print('\n  ℹ️  No brain-specific tissues labelled — showing top 20 tissues')
    display(df_exp.head(20).to_string(index=False))

# Top 20 bar chart
df_top20 = df_exp.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#E53935' if any(k in t.lower() for k in brain_keywords) else '#42A5F5'
              for t in df_top20['Tissue']]
ax.barh(df_top20['Tissue'], df_top20['RNA Value'], color=bar_colors)
ax.set_xlabel('RNA Expression (TPM)', fontsize=11)
ax.set_title(f'{GENE_SYMBOL} Expression Profile — Top 20 Tissues', fontsize=13, fontweight='bold')
red_patch  = mpatches.Patch(color='#E53935', label='Brain-related')
blue_patch = mpatches.Patch(color='#42A5F5', label='Other tissues')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.show()

---
## 📚 Section 4 — PubTator3: Gene–Disease Co-mentions & Publication Velocity

In [ ]:
def pubtator_search(query: str, page: int = 1, sort: str = 'date desc') -> dict:
    """Search PubTator3 for gene-disease co-mentions."""
    url = f'{PUBTATOR_API}/search/'
    params = {'text': query, 'sort': sort, 'page': page}
    try:
        r = requests.get(url, params=params, timeout=20)
        if r.status_code == 200:
            return r.json()
    except Exception as e:
        print(f'  ⚠️  PubTator error: {e}')
    return {}


def pubtator_export(pmids: list) -> list:
    """Export PubTator3 annotations for a list of PMIDs."""
    docs = []
    batch_size = 10
    for i in range(0, min(len(pmids), 100), batch_size):
        batch = pmids[i:i+batch_size]
        url = f'{PUBTATOR_API}/publications/export/biocjson'
        params = {'pmids': ','.join(str(p) for p in batch), 'full': 'true'}
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                d = r.json()
                if isinstance(d, list):
                    docs.extend(d)
                elif 'PubTator3' in d:
                    docs.extend(d['PubTator3'])
        except Exception as e:
            print(f'  ⚠️  Export error batch {i}: {e}')
        time.sleep(0.3)
    return docs


# ── Step 1: Get PMIDs for PHGDH + Alzheimer ──────────────────────────────────
query_str = f'{GENE_SYMBOL} {DISEASE_NAME}'
print(f'🔍 Searching PubTator3: "{query_str}" ...')

all_pmids = []
all_results_meta = []

for pg in range(1, 6):   # up to 5 pages
    res = pubtator_search(query_str, page=pg)
    hits = res.get('results', [])
    if not hits:
        break
    for h in hits:
        all_pmids.append(str(h.get('pmid', '')))
        all_results_meta.append({
            'PMID'    : str(h.get('pmid', '')),
            'Title'   : h.get('title', 'N/A'),
            'Journal' : h.get('journal', 'N/A'),
            'Year'    : str(h.get('date', ''))[:4] if h.get('date') else 'N/A',
        })
    total_count = res.get('count', 0)
    print(f'  Page {pg}: {len(hits)} results | Total reported: {total_count}')
    time.sleep(0.5)

unique_pmids = list(dict.fromkeys(all_pmids))   # preserve order, deduplicate
print(f'\n  📊 Unique PMIDs collected: {len(unique_pmids)}')

# ── Step 2: Show top papers ───────────────────────────────────────────────────
df_pt_papers = pd.DataFrame(all_results_meta).drop_duplicates('PMID')
print(f'\n📄 Top PHGDH × Alzheimer Papers from PubTator3:')
display(df_pt_papers.head(20))

In [ ]:
# ── Step 3: Publication velocity by year ──────────────────────────────────────
year_counts = df_pt_papers[df_pt_papers['Year'].str.match(r'^\d{4}$', na=False)]['Year'].value_counts().sort_index()

if not year_counts.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Bar chart — publications per year
    axes[0].bar(year_counts.index, year_counts.values, color='#5C6BC0', edgecolor='white')
    axes[0].set_title(f'{GENE_SYMBOL} × Alzheimer: Publications per Year\n(PubTator3)', fontweight='bold')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Publications')
    axes[0].tick_params(axis='x', rotation=45)

    # Cumulative trend
    cum = year_counts.cumsum()
    axes[1].plot(cum.index, cum.values, marker='o', color='#E53935', linewidth=2)
    axes[1].fill_between(cum.index, cum.values, alpha=0.15, color='#E53935')
    axes[1].set_title(f'Cumulative Publications\n(PubTator3)', fontweight='bold')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Cumulative Count')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    # Velocity stats
    total_pt = len(unique_pmids)
    import datetime
    current_year = datetime.datetime.now().year
    recent_pt = year_counts[year_counts.index.astype(int) >= current_year - 2].sum()
    velocity_pt = (recent_pt / total_pt * 100) if total_pt > 0 else 0
    print(f'\n📈 PubTator3 Velocity Stats for {GENE_SYMBOL} × Alzheimer:')
    print(f'   Total papers matched : {total_pt}')
    print(f'   Recent (last 3 yrs)  : {recent_pt}')
    print(f'   Velocity             : {velocity_pt:.1f}%')
else:
    print('  ℹ️  Not enough year data to plot velocity')

In [ ]:
# ── Step 4: Extract co-mentioned genes from PubTator annotations ─────────────
BLACKLIST = {'TNM','PCR','DNA','RNA','MRI','CT','PET','IHC','BMI','AUC','ROC',
             'HER','CI','HR','OR','SD','CR','ER','PFS','DFS','RCT','FDA','WHO',
             'CRP','LDH','PSA','CBC','ECG','EEG','CEA','AFP','HCG','CA'}

print(f'\n🧬 Extracting gene annotations from {min(len(unique_pmids), 100)} PubMed articles...')
docs = pubtator_export(unique_pmids[:100])

gene_counts = {}
for doc in docs:
    for passage in doc.get('passages', []):
        for anno in passage.get('annotations', []):
            if anno.get('infons', {}).get('type') != 'Gene':
                continue
            identifier = anno.get('infons', {}).get('identifier', '')
            if not identifier or not identifier.isdigit():
                continue
            gene_name = anno.get('infons', {}).get('name') or anno.get('text', '')
            if not gene_name:
                continue
            import re
            if not re.match(r'^[A-Za-z][A-Za-z0-9]{1,9}$', gene_name):
                continue
            if gene_name.upper() in BLACKLIST:
                continue
            g = gene_name.upper()
            gene_counts[g] = gene_counts.get(g, 0) + 1

top_genes = sorted(gene_counts.items(), key=lambda x: x[1], reverse=True)[:30]
print(f'  ✅ Unique genes extracted: {len(gene_counts)}')

if top_genes:
    df_genes = pd.DataFrame(top_genes, columns=['Gene', 'Mentions'])
    print(f'\n🧬 Top Co-mentioned Genes with PHGDH in Alzheimer literature:')
    display(df_genes.head(20).to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 6))
    phgdh_color = ['#E53935' if g == 'PHGDH' else '#42A5F5' for g in df_genes['Gene']]
    ax.barh(df_genes['Gene'], df_genes['Mentions'], color=phgdh_color)
    ax.set_xlabel('PubTator3 Mentions')
    ax.set_title(f'Top Co-mentioned Genes: {GENE_SYMBOL} × {DISEASE_NAME}', fontweight='bold')
    red_p  = mpatches.Patch(color='#E53935', label=f'{GENE_SYMBOL} (query gene)')
    blue_p = mpatches.Patch(color='#42A5F5', label='Co-mentioned genes')
    ax.legend(handles=[red_p, blue_p])
    plt.tight_layout()
    plt.show()
else:
    print('  ℹ️  No annotated genes found in export — PubTator3 may have limited coverage for this query')

---
## 📰 Section 5 — Europe PMC: Literature Analysis

In [ ]:
import datetime
current_year = datetime.datetime.now().year
three_yrs_ago = current_year - 3

def epmc_search(query: str, page_size: int = 25, cursor_mark: str = '*') -> dict:
    params = {
        'query'     : query,
        'format'    : 'json',
        'pageSize'  : page_size,
        'cursorMark': cursor_mark,
        'sort'      : 'P_PDATE_D desc'
    }
    try:
        r = requests.get(EPMC_API, params=params, timeout=20)
        return r.json() if r.status_code == 200 else {}
    except Exception as e:
        print(f'  ⚠️  Europe PMC error: {e}')
        return {}


# Total papers
q_total   = f'{GENE_SYMBOL} AND "{DISEASE_NAME}"'
q_recent  = f'{GENE_SYMBOL} AND "{DISEASE_NAME}" AND FIRST_PDATE:[{three_yrs_ago}-01-01 TO {current_year}-12-31]'
q_brain   = f'{GENE_SYMBOL} AND "{DISEASE_NAME}" AND (brain OR neuron OR neurodegeneration)'

print(f'🔍 Querying Europe PMC...')
res_total  = epmc_search(q_total,  page_size=25)
res_recent = epmc_search(q_recent, page_size=25)
res_brain  = epmc_search(q_brain,  page_size=25)

total_count  = res_total.get('hitCount', 0)
recent_count = res_recent.get('hitCount', 0)
brain_count  = res_brain.get('hitCount', 0)
velocity_pct = (recent_count / total_count * 100) if total_count > 0 else 0

print(f'\n📊 Europe PMC Stats for {GENE_SYMBOL} × {DISEASE_NAME}:')
print(f'   Total papers          : {total_count}')
print(f'   Recent (last 3 yrs)   : {recent_count}')
print(f'   Brain/neuron subset   : {brain_count}')
print(f'   Velocity              : {velocity_pct:.1f}%')

# ── Top papers from Europe PMC ───────────────────────────────────────────────
top_epmc_papers = []
for r in res_total.get('resultList', {}).get('result', []):
    top_epmc_papers.append({
        'Title'  : r.get('title', 'N/A'),
        'Journal': r.get('journalTitle', r.get('journalInfo', {}).get('journal', {}).get('title', 'N/A')),
        'Year'   : r.get('pubYear', 'N/A'),
        'PMID'   : r.get('pmid', 'N/A'),
        'DOI'    : r.get('doi', 'N/A'),
        'Citations': r.get('citedByCount', 0)
    })

df_epmc = pd.DataFrame(top_epmc_papers)
print(f'\n📄 Top Papers from Europe PMC:')
if not df_epmc.empty:
    display(df_epmc)
else:
    print('  ℹ️  No papers returned — PHGDH × Alzheimer is an emerging area')

In [ ]:
# ── Fetch all PMIDs for year-trend plot ───────────────────────────────────────
print('📈 Fetching year distribution from Europe PMC...')
all_epmc_years = []
cursor = '*'

for _ in range(10):   # max 250 records
    res = epmc_search(q_total, page_size=25, cursor_mark=cursor)
    papers = res.get('resultList', {}).get('result', [])
    if not papers:
        break
    for p in papers:
        yr = p.get('pubYear')
        if yr and str(yr).isdigit():
            all_epmc_years.append(int(yr))
    next_cursor = res.get('nextCursorMark')
    if not next_cursor or next_cursor == cursor:
        break
    cursor = next_cursor
    time.sleep(0.3)

if all_epmc_years:
    yr_series = pd.Series(all_epmc_years).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(10, 4))
    bar_colors = ['#E53935' if y >= three_yrs_ago else '#42A5F5' for y in yr_series.index]
    ax.bar(yr_series.index, yr_series.values, color=bar_colors)
    ax.set_title(f'{GENE_SYMBOL} × Alzheimer: Publications per Year (Europe PMC)', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Publications')
    red_p  = mpatches.Patch(color='#E53935', label=f'Recent ({three_yrs_ago}–{current_year})')
    blue_p = mpatches.Patch(color='#42A5F5', label='Earlier')
    ax.legend(handles=[red_p, blue_p])
    plt.tight_layout()
    plt.show()
else:
    print('  ℹ️  Not enough data for year trend')

---
## 🏥 Section 6 — ClinicalTrials.gov: Trial Landscape

In [ ]:
def fetch_clinical_trials(gene: str, disease: str, page_size: int = 100) -> dict:
    """ClinicalTrials.gov API v2 — mirrors getDrillDownData logic from api.ts."""
    fields = ','.join([
        'protocolSection.identificationModule.nctId',
        'protocolSection.identificationModule.briefTitle',
        'protocolSection.statusModule.overallStatus',
        'protocolSection.designModule.phases',
        'protocolSection.designModule.studyType',
        'protocolSection.conditionsModule.conditions',
        'protocolSection.armsInterventionsModule.interventions',
        'protocolSection.sponsorCollaboratorsModule.leadSponsor.class',
        'protocolSection.statusModule.startDateStruct',
        'protocolSection.statusModule.primaryCompletionDateStruct',
    ])
    params = {
        'query.cond' : disease,
        'query.term' : gene,
        'pageSize'   : page_size,
        'countTotal' : 'true',
        'fields'     : fields
    }
    try:
        r = requests.get(CT_API, params=params, timeout=20)
        return r.json() if r.status_code == 200 else {}
    except Exception as e:
        print(f'  ⚠️  ClinicalTrials error: {e}')
        return {}


print(f'🔍 Querying ClinicalTrials.gov for {GENE_SYMBOL} × {DISEASE_NAME}...')
ct_data = fetch_clinical_trials(GENE_SYMBOL, DISEASE_NAME)

# Fallback: broader search
if not ct_data.get('totalCount'):
    print('  ↩️  No results with disease filter — broadening to gene-only search...')
    ct_data = fetch_clinical_trials(GENE_SYMBOL, '')

studies   = ct_data.get('studies', [])
total_ct  = ct_data.get('totalCount', 0)
print(f'  ✅ Total trials found: {total_ct}')

# ── Parse studies ─────────────────────────────────────────────────────────────
PHASE_ORDER  = ['EARLY_PHASE1', 'PHASE1', 'PHASE2', 'PHASE3', 'PHASE4']
ACTIVE_STATS = {'RECRUITING', 'ACTIVE_NOT_RECRUITING', 'ENROLLING_BY_INVITATION'}

phase_counts = {p: 0 for p in PHASE_ORDER}
sponsor_counts = {}
status_counts  = {}
max_phase_idx  = -1
trial_rows = []

for s in studies:
    ps   = s.get('protocolSection', {})
    ident = ps.get('identificationModule', {})
    stat  = ps.get('statusModule', {})
    design= ps.get('designModule', {})
    sponsor = ps.get('sponsorCollaboratorsModule', {}).get('leadSponsor', {})

    nct    = ident.get('nctId', 'N/A')
    title  = ident.get('briefTitle', 'N/A')
    status = stat.get('overallStatus', 'UNKNOWN')
    phases = design.get('phases', [])
    stype  = design.get('studyType', 'N/A')
    sclass = sponsor.get('class', 'N/A')

    status_counts[status] = status_counts.get(status, 0) + 1
    sclass_label = 'NIH' if sclass in ('NIH','FED','U_S_FED') else sclass
    sponsor_counts[sclass_label] = sponsor_counts.get(sclass_label, 0) + 1

    for p in phases:
        idx = PHASE_ORDER.index(p) if p in PHASE_ORDER else -1
        if idx > max_phase_idx:
            max_phase_idx = idx
        if p in phase_counts:
            phase_counts[p] += 1

    trial_rows.append({
        'NCT ID'  : nct,
        'Title'   : title[:80] + '...' if len(title) > 80 else title,
        'Status'  : status,
        'Phases'  : ', '.join(phases) if phases else 'N/A',
        'Type'    : stype,
        'Sponsor' : sclass_label
    })

max_phase = PHASE_ORDER[max_phase_idx] if max_phase_idx >= 0 else 'N/A'
active_count = sum(1 for s in studies if
    s.get('protocolSection', {}).get('statusModule', {}).get('overallStatus') in ACTIVE_STATS)

print(f'\n🏥 ClinicalTrials.gov Summary:')
print(f'   Total trials       : {total_ct}')
print(f'   Max phase reached  : {max_phase}')
print(f'   Active trials      : {active_count}')
print(f'   Phase breakdown    : {phase_counts}')
print(f'   Sponsor breakdown  : {sponsor_counts}')

In [ ]:
# ── Clinical Trials visualizations ───────────────────────────────────────────
df_trials = pd.DataFrame(trial_rows)

if not df_trials.empty:
    print(f'\n📋 Clinical Trial Listing ({len(df_trials)} shown):')
    display(df_trials)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Phase breakdown
    non_zero_phases = {k: v for k, v in phase_counts.items() if v > 0}
    if non_zero_phases:
        axes[0].bar(non_zero_phases.keys(), non_zero_phases.values(), color='#5C6BC0')
        axes[0].set_title('Trials by Phase', fontweight='bold')
        axes[0].set_xlabel('Phase')
        axes[0].set_ylabel('Count')
        axes[0].tick_params(axis='x', rotation=30)
    else:
        axes[0].text(0.5, 0.5, 'No phase data', ha='center', va='center')
        axes[0].set_title('Trials by Phase', fontweight='bold')

    # Status distribution
    if status_counts:
        axes[1].bar(status_counts.keys(), status_counts.values(), color='#26A69A')
        axes[1].set_title('Trials by Status', fontweight='bold')
        axes[1].set_xlabel('Status')
        axes[1].set_ylabel('Count')
        axes[1].tick_params(axis='x', rotation=30)

    # Sponsor breakdown
    if sponsor_counts:
        axes[2].pie(sponsor_counts.values(), labels=sponsor_counts.keys(),
                    autopct='%1.1f%%', colors=sns.color_palette('Set2'))
        axes[2].set_title('Trials by Sponsor Class', fontweight='bold')

    plt.suptitle(f'ClinicalTrials.gov: {GENE_SYMBOL} × {DISEASE_NAME}', y=1.01,
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'  ℹ️  No clinical trials found for {GENE_SYMBOL} × {DISEASE_NAME}')
    print(f'     This is consistent with PHGDH being an emerging/pre-clinical target')

---
## 📊 Section 7 — Summary Dashboard: All Scores at a Glance

In [ ]:
# ── Collect all stats ─────────────────────────────────────────────────────────
summary = {
    '── GENE INFO ──'                 : '',
    'Gene Symbol'                     : GENE_SYMBOL,
    'Ensembl ID'                      : GENE_ENSEMBL,
    'Gene Name'                       : target_data.get('approvedName', 'N/A'),
    'Biotype'                         : target_data.get('biotype', 'N/A'),
    '── OPEN TARGETS SCORES ──'       : '',
    'OT Overall Association Score'    : overall_score if overall_score else 'Not in top 300',
    'G — Genetic Association Score'   : round(G, 4),
    'E — Expression Score'            : round(E, 4),
    'T — Tractability Score'          : round(T, 4),
    'GET Score (G×0.5 + E×0.25 + T×0.25)': round(GET, 4),
    'Tractability Tier'               : tier_map.get(T, 'Unknown'),
    '── LITERATURE ──'                : '',
    'Europe PMC — Total Papers'       : total_count,
    'Europe PMC — Recent (3yr)'       : recent_count,
    'Europe PMC — Brain subset'       : brain_count,
    'Europe PMC — Velocity'           : f'{velocity_pct:.1f}%',
    'PubTator3 — Total PMIDs'         : len(unique_pmids),
    'PubTator3 — Recent velocity'     : f'{velocity_pt:.1f}%' if 'velocity_pt' in dir() else 'N/A',
    '── CLINICAL TRIALS ──'           : '',
    'ClinicalTrials — Total'          : total_ct,
    'ClinicalTrials — Max Phase'      : max_phase,
    'ClinicalTrials — Active Trials'  : active_count,
}

print('=' * 60)
print(f'  PHGDH × ALZHEIMER\'S DISEASE — FULL SUMMARY')
print('=' * 60)
for k, v in summary.items():
    if '──' in k:
        print(f'\n{k}')
    else:
        print(f'  {k:<42} {v}')
print('=' * 60)

In [ ]:
# ── Radar / Spider chart — GET score components ───────────────────────────────
import numpy as np

categories = ['Genetic\nAssociation (G)', 'Expression (E)', 'Tractability (T)',
              'Literature\nVelocity', 'Clinical\nEvidence']

# Normalize literature velocity to 0-1
lit_velocity_norm = min(1.0, velocity_pct / 100)
# Normalize clinical evidence: 0 trials → 0.0, any trial → scale by phase
phase_to_score = {'N/A': 0.0, 'EARLY_PHASE1': 0.15, 'PHASE1': 0.30,
                  'PHASE2': 0.55, 'PHASE3': 0.75, 'PHASE4': 1.0}
clinical_norm = phase_to_score.get(max_phase, 0.0)

values = [G, E, T, lit_velocity_norm, clinical_norm]
values_closed = values + [values[0]]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles_closed = angles + [angles[0]]
labels_closed = categories + [categories[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})
ax.plot(angles_closed, values_closed, 'o-', linewidth=2, color='#E53935')
ax.fill(angles_closed, values_closed, alpha=0.25, color='#E53935')
ax.set_thetagrids(np.degrees(angles), categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title(f'{GENE_SYMBOL} × Alzheimer Disease\nTarget Scoring Profile\nGET = {GET:.3f}',
             fontsize=13, fontweight='bold', pad=20)

# Annotate values
for angle, val, label in zip(angles, values, categories):
    ax.annotate(f'{val:.3f}',
                xy=(angle, val),
                xytext=(angle, val + 0.07),
                ha='center', va='center',
                fontsize=9, fontweight='bold', color='#B71C1C')

plt.tight_layout()
plt.show()
print(f'\n✅ Analysis complete for {GENE_SYMBOL} × {DISEASE_NAME}')
print(f'   GET Score = {GET:.4f}')
print(f'   Tractability: {tier_map.get(T)}')
print(f'   Clinical stage: {max_phase}')

---
## 📝 Notes & Interpretation Guide

### GET Score thresholds (from api.ts)
| GET Score | Interpretation |
|-----------|---------------|
| > 0.60    | Strong candidate — genetic + druggable |
| 0.40–0.60 | Moderate candidate — some evidence |
| 0.20–0.40 | Weak / emerging — early-stage signal |
| < 0.20    | Speculative — data-limited |

### Tractability tiers (T score)
| T Score | Tier |
|---------|------|
| 1.00 | Approved drug exists |
| 0.85 | Advanced clinical evidence |
| 0.70 | Phase 1 clinical evidence |
| 0.55 | Structural ligand data |
| 0.40 | Druggable pocket (computational) |
| 0.25 | Druggable family membership |
| 0.10 | No tractability evidence |

### Data Sources
- **Open Targets**: https://platform.opentargets.org/target/ENSG00000092621
- **PubTator3**: https://www.ncbi.nlm.nih.gov/research/pubtator3/
- **Europe PMC**: https://europepmc.org/
- **ClinicalTrials.gov**: https://clinicaltrials.gov/
- **Cell 2025 paper context**: PHGDH — serine biosynthesis pathway enzyme, emerging role in neurodegeneration
